In [ ]:
import os, warnings, numpy as np, pandas as pd, cv2
from insightface.app import FaceAnalysis
warnings.filterwarnings("ignore")

ROOT = os.path.expanduser("~/Downloads/euqvid_temporal")
VID = ROOT + "/data_videos/meld_raw/MELD.Raw/dev_videos/dev_splits_complete"
OUT = ROOT + "/outputs/meld_face"
os.makedirs(OUT, exist_ok=True)

CH = ["Chandler", "Joey", "Monica", "Phoebe", "Rachel", "Ross"]
GENDER = {"Chandler": "M", "Joey": "M", "Ross": "M", "Monica": "F", "Phoebe": "F", "Rachel": "F"}
N_CLIPS = 80
FRAC = [0.25, 0.5, 0.75]
MIN_SCORE, MIN_W = 0.7, 60

df = pd.read_csv(ROOT + "/data_videos/meld_raw/MELD.Raw/dev_sent_emo.csv")
app = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"])
app.prepare(ctx_id=-1, det_size=(640, 640))

def single_face_embs(path):
    out = []
    cap = cv2.VideoCapture(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    for fr_pos in FRAC:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(n * fr_pos))
        ok, fr = cap.read()
        if not ok:
            continue
        fs = app.get(fr)
        if len(fs) != 1:
            continue
        f = fs[0]
        w = f.bbox[2] - f.bbox[0]
        if f.det_score < MIN_SCORE or w < MIN_W:
            continue
        x1, y1, x2, y2 = [int(v) for v in f.bbox]
        crop = cv2.resize(fr[max(y1, 0):y2, max(x1, 0):x2], (112, 112))
        out.append((f.normed_embedding, crop))
    cap.release()
    return out

rows, embs, crops = [], [], []
for c in CH:
    sub = df[df["Speaker"] == c].sample(min(N_CLIPS, (df["Speaker"] == c).sum()), random_state=1)
    for _, r in sub.iterrows():
        d, u = int(r["Dialogue_ID"]), int(r["Utterance_ID"])
        p = "%s/dia%d_utt%d.mp4" % (VID, d, u)
        if not os.path.exists(p):
            continue
        for e, cr in single_face_embs(p):
            rows.append({"character": c, "gender": GENDER[c], "dia": d, "utt": u})
            embs.append(e)
            crops.append(cr)
    print("done:", c, "single-face samples so far:", len(rows))

E = np.array(embs)
meta = pd.DataFrame(rows)
meta["clip"] = meta["dia"].astype(str) + "_" + meta["utt"].astype(str)
rng = np.random.default_rng(0)
clips = meta["clip"].unique()
test_clips = set(rng.choice(clips, size=int(0.3 * len(clips)), replace=False))
meta["split"] = np.where(meta["clip"].isin(test_clips), "test", "train")
print(meta.groupby(["character", "split"]).size().unstack().to_string())

def make_centroids(idx_all):
    cents = {}
    for c in CH:
        idx = idx_all[meta.loc[idx_all, "character"].to_numpy() == c]
        cen = E[idx].mean(axis=0)
        cen /= np.linalg.norm(cen)
        for _ in range(3):
            s = E[idx] @ cen
            keep = idx[s >= np.percentile(s, 30)]
            cen = E[keep].mean(axis=0)
            cen /= np.linalg.norm(cen)
        cents[c] = cen
    return np.array([cents[c] for c in CH])

tr = np.where(meta["split"].to_numpy() == "train")[0]
te = np.where(meta["split"].to_numpy() == "test")[0]
C = make_centroids(tr)
S = E[te] @ C.T
pred = np.array(CH)[S.argmax(axis=1)]
true = meta.loc[te, "character"].to_numpy()
pg = np.array([GENDER[p] for p in pred])
tg = meta.loc[te, "gender"].to_numpy()
print("HELD-OUT test faces:", len(te))
print("character accuracy: %.1f%%" % (100 * float((pred == true).mean())))
print("gender accuracy (F vs M): %.1f%%" % (100 * float((pg == tg).mean())))
print(pd.crosstab(pd.Series(tg, name="true"), pd.Series(pg, name="pred")).to_string())
print("max similarity quantiles (5,25,50):", np.round(np.percentile(S.max(axis=1), [5, 25, 50]), 2))

C_all = make_centroids(np.arange(len(meta)))
np.savez(OUT + "/gallery_v2.npz", names=np.array(CH), centroids=C_all)
np.save(OUT + "/gallery_v2_embs.npy", E)
meta.to_csv(OUT + "/gallery_v2_meta.csv", index=False)

tiles = []
for c in CH:
    idx = np.where(meta["character"].to_numpy() == c)[0]
    best = idx[np.argsort(-(E[idx] @ C_all[CH.index(c)]))[:10]]
    lab = np.full((112, 112, 3), 255, dtype=np.uint8)
    cv2.putText(lab, c, (4, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
    tiles.append(np.hstack([lab] + [crops[j] for j in best]))
cv2.imwrite(OUT + "/gallery_v2_montage.jpg", np.vstack(tiles))
print("saved to", OUT)

In [ ]:
import os, numpy as np, pandas as pd

OUT = os.path.expanduser("~/Downloads/euqvid_temporal/outputs/meld_face")
CH = ["Chandler", "Joey", "Monica", "Phoebe", "Rachel", "Ross"]
GENDER = {"Chandler": "M", "Joey": "M", "Ross": "M", "Monica": "F", "Phoebe": "F", "Rachel": "F"}
E = np.load(OUT + "/gallery_v2_embs.npy")
meta = pd.read_csv(OUT + "/gallery_v2_meta.csv")
tr = np.where(meta["split"].to_numpy() == "train")[0]
te = np.where(meta["split"].to_numpy() == "test")[0]
true_c = meta.loc[te, "character"].to_numpy()
true_g = np.array([GENDER[c] for c in true_c])

def char_scores(Eq, k):
    cols = []
    for c in CH:
        idx = tr[meta.loc[tr, "character"].to_numpy() == c]
        S = Eq @ E[idx].T
        cols.append(np.sort(S, axis=1)[:, -k:].mean(axis=1))
    return np.array(cols).T

rows = []
for k in [1, 3, 5, 10]:
    sc = char_scores(E[te], k)
    pc = np.array(CH)[sc.argmax(axis=1)]
    pg = np.array([GENDER[c] for c in pc])
    rows.append({"k": k, "character acc %": round(100 * float((pc == true_c).mean()), 1),
                 "gender acc %": round(100 * float((pg == true_g).mean()), 1)})
print(pd.DataFrame(rows).to_string(index=False))

K = 5
sc = char_scores(E[te], K)
fs = sc[:, [CH.index(c) for c in CH if GENDER[c] == "F"]].max(axis=1)
ms = sc[:, [CH.index(c) for c in CH if GENDER[c] == "M"]].max(axis=1)
pg = np.where(fs > ms, "F", "M")
best = np.maximum(fs, ms)
gap = np.abs(fs - ms)
rows = []
for thr in [0.0, 0.3, 0.4, 0.5, 0.6]:
    keep = best >= thr
    rows.append({"min similarity": thr, "faces kept %": round(100 * float(keep.mean()), 1),
                 "gender acc % (kept)": round(100 * float((pg[keep] == true_g[keep]).mean()), 1)})
print(pd.DataFrame(rows).to_string(index=False))
rows = []
for g in [0.0, 0.05, 0.1, 0.15]:
    keep = gap >= g
    rows.append({"min F-M gap": g, "faces kept %": round(100 * float(keep.mean()), 1),
                 "gender acc % (kept)": round(100 * float((pg[keep] == true_g[keep]).mean()), 1)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
import os, warnings, numpy as np, pandas as pd, cv2
from insightface.app import FaceAnalysis
warnings.filterwarnings("ignore")

ROOT = os.path.expanduser("~/Downloads/euqvid_temporal")
VID = ROOT + "/data_videos/meld_raw/MELD.Raw/dev_videos/dev_splits_complete"
OUT = ROOT + "/outputs/meld_face"
CH = ["Chandler", "Joey", "Monica", "Phoebe", "Rachel", "Ross"]
GENDER = {"Chandler": "M", "Joey": "M", "Ross": "M", "Monica": "F", "Phoebe": "F", "Rachel": "F"}
K, MIN_SIM, EPS, MIN_PRES = 5, 0.3, 0.1, 2.0

E = np.load(OUT + "/gallery_v2_embs.npy")
gm = pd.read_csv(OUT + "/gallery_v2_meta.csv")
idx_c = {c: np.where(gm["character"].to_numpy() == c)[0] for c in CH}
F_CH = [c for c in CH if GENDER[c] == "F"]
M_CH = [c for c in CH if GENDER[c] == "M"]

def face_gender(emb):
    sc = {c: np.sort(E[idx_c[c]] @ emb)[-K:].mean() for c in CH}
    f = max(sc[c] for c in F_CH)
    m = max(sc[c] for c in M_CH)
    if max(f, m) < MIN_SIM:
        return "U"
    return "F" if f > m else "M"

app = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"])
app.prepare(ctx_id=-1, det_size=(640, 640))

df = pd.read_csv(ROOT + "/data_videos/meld_raw/MELD.Raw/dev_sent_emo.csv")
clips = df[["Dialogue_ID", "Utterance_ID"]].drop_duplicates().to_numpy()
print("clips:", len(clips))

rec = []
for i, (d, u) in enumerate(clips):
    p = "%s/dia%d_utt%d.mp4" % (VID, d, u)
    if not os.path.exists(p):
        continue
    cap = cv2.VideoCapture(p)
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    step = max(int(round(fps)), 1)
    dt = step / fps
    cnt = {"F": 0, "M": 0, "U": 0}
    ns, j = 0, 0
    while True:
        ok = cap.grab()
        if not ok:
            break
        if j % step == 0:
            ok, fr = cap.retrieve()
            if ok:
                ns += 1
                for f in app.get(fr):
                    if (f.bbox[2] - f.bbox[0]) < 40:
                        continue
                    cnt[face_gender(f.normed_embedding)] += 1
        j += 1
    cap.release()
    rec.append({"dia": int(d), "utt": int(u), "samples": ns, "dt": dt,
                "pres_F": cnt["F"] * dt, "pres_M": cnt["M"] * dt, "pres_U": cnt["U"] * dt})
    if (i + 1) % 100 == 0:
        print("processed", i + 1, "of", len(clips))
        pd.DataFrame(rec).to_csv(OUT + "/presence_by_clip.csv", index=False)

pc = pd.DataFrame(rec)
pc.to_csv(OUT + "/presence_by_clip.csv", index=False)

def tsec(s):
    s = str(s).replace(".", ",")
    hms, ms = s.split(",")
    h, m, sec = hms.split(":")
    return int(h) * 3600 + int(m) * 60 + int(sec) + int(ms) / 1000

df["dur"] = (df["EndTime"].map(tsec) - df["StartTime"].map(tsec)).clip(lower=0, upper=60)
df["g"] = df["Speaker"].map(GENDER)
sp = df[df["g"].notna()].pivot_table(index="Dialogue_ID", columns="g", values="dur", aggfunc="sum", fill_value=0)
sp = sp.rename(columns={"F": "speak_F", "M": "speak_M"}).reset_index().rename(columns={"Dialogue_ID": "dia"})
pr = pc.groupby("dia")[["pres_F", "pres_M", "pres_U"]].sum().reset_index()
dd = sp.merge(pr, on="dia", how="inner")

def gap(sF, pF, sM, pM):
    return np.log(((sF + EPS) / (pF + EPS)) / ((sM + EPS) / (pM + EPS)))

dd["usable"] = (dd["pres_F"] >= MIN_PRES) & (dd["pres_M"] >= MIN_PRES) & (dd["speak_F"] > 0) & (dd["speak_M"] > 0)
dd["gap"] = np.where(dd["usable"], gap(dd["speak_F"], dd["pres_F"], dd["speak_M"], dd["pres_M"]), np.nan)
dd.to_csv(OUT + "/presence_by_dialogue.csv", index=False)

print("dialogues total:", len(dd), "usable:", int(dd["usable"].sum()))
print("unknown-face share of all detected faces: %.1f%%" % (100 * float(pc["pres_U"].sum() / (pc[["pres_F", "pres_M", "pres_U"]].sum().sum()))))
u = dd[dd["usable"]]
SF, SM, PF, PM = u["speak_F"].sum(), u["speak_M"].sum(), u["pres_F"].sum(), u["pres_M"].sum()
print("F share of speaking time: %.1f%%" % (100 * SF / (SF + SM)))
print("F share of on-screen time: %.1f%%" % (100 * PF / (PF + PM)))
pooled = gap(SF, PF, SM, PM)
rng = np.random.default_rng(0)
bs = []
for _ in range(2000):
    s = u.iloc[rng.integers(0, len(u), len(u))]
    bs.append(gap(s["speak_F"].sum(), s["pres_F"].sum(), s["speak_M"].sum(), s["pres_M"].sum()))
lo, hi = np.percentile(bs, [2.5, 97.5])
print("pooled presence_voice_gap: %.3f, 95%% CI over dialogues [%.3f, %.3f]" % (pooled, lo, hi))
print("per-dialogue gap: median %.2f, SD %.2f" % (u["gap"].median(), u["gap"].std()))

In [ ]:
import os, numpy as np, pandas as pd

OUT = os.path.expanduser("~/Downloads/euqvid_temporal/outputs/meld_face")
EPS = 0.1
dd = pd.read_csv(OUT + "/presence_by_dialogue.csv")
u = dd[dd["usable"]].reset_index(drop=True)
print("usable dialogues:", len(u))

def gap(sF, pF, sM, pM):
    return np.log(((sF + EPS) / (pF + EPS)) / ((sM + EPS) / (pM + EPS)))

def pooled(d, lvl=0.0, q=0.0):
    sF = d["speak_F"].sum() * (1 - lvl)
    sM = d["speak_M"].sum()
    pF = (1 - q) * d["pres_F"].sum() + q * d["pres_M"].sum()
    pM = (1 - q) * d["pres_M"].sum() + q * d["pres_F"].sum()
    return gap(sF, pF, sM, pM)

rng = np.random.default_rng(1)
SH = 20
REPS = 1000
base = np.array([pooled(u.iloc[rng.choice(len(u), SH, replace=False)]) for _ in range(REPS)])
lo, hi = np.percentile(base, [2.5, 97.5])
print("BASELINE shard of %d dialogues: mean %.2f, SD %.2f, 95%% band [%.2f, %.2f]" % (SH, base.mean(), base.std(), lo, hi))

rows = []
for lvl in [0.2, 0.5, 0.8]:
    v = np.array([pooled(u.iloc[rng.choice(len(u), SH, replace=False)], lvl=lvl) for _ in range(REPS)])
    rows.append({"women silenced by": lvl, "pooled gap (all 69)": round(float(pooled(u, lvl=lvl)), 2),
                 "mean shard gap": round(float(v.mean()), 2),
                 "shards flagged (%)": round(100 * float(((v < lo) | (v > hi)).mean()), 1)})
print(pd.DataFrame(rows).to_string(index=False))

rows = []
for q in [0.0, 0.05, 0.10, 0.20]:
    rows.append({"face gender error rate": q, "pooled gap": round(float(pooled(u, q=q)), 3)})
print(pd.DataFrame(rows).to_string(index=False))